# Exercise 3 — Univariate Spectral Analysis: PSD Estimation from LFP

**Neural Data Science (WP7)** · LMU Biology

---

### Overview

Neural oscillations — rhythmic fluctuations in local field potentials (LFP) — are
cardinal signatures of brain network states.  Estimating the **power spectral
density (PSD)** is the first step in any frequency-domain analysis of neural data.

In this exercise you will:
1. Load hippocampal LFP data and explore its time-domain structure
2. Estimate the PSD using Welch's method (`scipy.signal.welch`)
3. Estimate the PSD using multitaper methods (`wp7_helpers.psd_multitaper`)
4. Explore the resolution–variance trade-off by varying NW

### Learning objectives

- Understand why raw periodograms are noisy and how averaging helps
- Compare Welch (windowed segments) and multitaper (DPSS tapers) approaches
- Interpret the time-halfbandwidth product NW and its effect on the spectrum
- Identify spectral features in hippocampal LFP (1/f, theta, gamma)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import scipy.signal
import sys
import os

# Add the course library to the path
sys.path.insert(0, '../../lib')
from wp7_helpers import psd_multitaper

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

### Load the data

We use a 16-channel hippocampal LFP recording (single shank) from the
ds-wp7 dataset, sampled at **1250 Hz**.

| Variable | Shape | Description |
|----------|-------|-------------|
| `lfps`   | `(n_samples, 16)` | LFP traces, one column per channel |

> **Note:** If `mne` is not installed, `wp7_helpers` automatically falls back
> to `scipy.signal` — the results are equivalent.  See `planning/env-audit-2026-04.md`
> for environment details.

In [ ]:
data_path = '../../data/spectral/ws_data_1shank.mat'
# Data is at ../../data/spectral/ relative to this exercise directory

if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Data not found at {data_path!r}. Adjust data_path to point to ws_data_1shank.mat"
    )

mat = scipy.io.loadmat(data_path)
lfps = mat['lfps']
fs = 1250  # Hz

n_samples, n_channels = lfps.shape
duration_sec = n_samples / fs
print(f'LFP shape: {lfps.shape}')
print(f'  {n_channels} channels, {n_samples:,} samples, {duration_sec:.1f} s at {fs} Hz')

---
## Section 1 — Exploring the LFP Time Series

Before computing any spectrum, look at the raw signal.  LFP is a
continuous voltage trace — you should be able to see oscillatory
activity by eye in a short segment.

> **Think about it:** What frequency components can you identify visually?
> What is the dominant rhythm in a ~2 s segment of hippocampal LFP?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Plot a short LFP segment
# ──────────────────────────────────────────────────────
# 1. Select channel 0
# 2. Extract the first 5 seconds of data
# 3. Create a time axis in seconds
# 4. Plot the trace with labeled axes
# ──────────────────────────────────────────────────────

ch = 0
seg_sec = 5.0

raise NotImplementedError("Plot the LFP time series")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

To extract the first N seconds, you need `int(seg_sec * fs)` samples.
The time axis is `np.arange(n_seg_samples) / fs` in seconds.
Use `ax.plot(t, x)` for a simple line plot.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
seg_samples = int(seg_sec * fs)
x = lfps[:seg_samples, ch]
t = np.arange(len(x)) / fs
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
seg_samples = int(seg_sec * fs)
x = lfps[:seg_samples, ch]
t = np.arange(len(x)) / fs

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, x, linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (a.u.)')
ax.set_title(f'LFP — Channel {ch}, first {seg_sec} s')
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 1
assert 'x' in dir() or 'x' in locals(), "Variable `x` not found"
assert len(x) == int(seg_sec * fs), f"Expected {int(seg_sec * fs)} samples, got {len(x)}"
print(f"Checkpoint 1 passed — {len(x)} samples extracted")

---
## Section 2 — Welch PSD Estimation

Welch's method splits the signal into overlapping windowed segments,
computes a periodogram for each, and averages them.  This reduces
variance compared to a single periodogram, at the cost of frequency
resolution (set by the segment length).

> **Think about it:** If you double the segment length in Welch, what
> happens to the frequency resolution?  What happens to the number of
> segments that get averaged (and thus the variance)?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compute and plot the Welch PSD
# ──────────────────────────────────────────────────────
# 1. Use the FULL signal for this channel (all samples)
# 2. Call scipy.signal.welch(x_long, fs=fs, nperseg=...)
#    with a 2-second segment length
# 3. Plot the PSD on a log scale (semilogy)
#    with frequency limited to 0–200 Hz
# ──────────────────────────────────────────────────────

x_long = lfps[:, ch]  # full signal

raise NotImplementedError("Compute the Welch PSD and plot it")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`scipy.signal.welch` returns `(frequencies, power_spectral_density)`.
The `nperseg` parameter sets the segment length in **samples**.
For a 2-second window at 1250 Hz, that's `int(2.0 * 1250) = 2500`.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
nperseg = int(2.0 * fs)
freqs_welch, psd_welch = scipy.signal.welch(x_long, fs=fs, nperseg=nperseg)
```

Use `ax.semilogy(freqs, psd)` for a log-scale y-axis.
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
nperseg = int(2.0 * fs)
freqs_welch, psd_welch = scipy.signal.welch(x_long, fs=fs, nperseg=nperseg)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs_welch, psd_welch, linewidth=0.8)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (a.u.² / Hz)')
ax.set_title('Welch PSD — Channel 0')
ax.set_xlim(0, 200)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 2
assert len(freqs_welch) > 10, "Too few frequency bins"
assert np.all(psd_welch >= 0), "PSD should be non-negative"
assert freqs_welch[-1] <= fs / 2 + 1, "Frequencies exceed Nyquist"
print(f"Checkpoint 2 passed — Welch PSD: {len(freqs_welch)} freq bins, "
      f"max freq = {freqs_welch[-1]:.0f} Hz")

---
## Section 3 — Multitaper PSD and the NW Trade-Off

Multitaper estimation uses **DPSS (Slepian) tapers** that are optimally
concentrated in frequency.  The key parameter is **NW** (time-halfbandwidth
product): it controls how many tapers are used (K = 2·NW − 1) and the
resulting trade-off between spectral resolution and variance.

- **Low NW** (e.g., 2): 3 tapers, sharp peaks, higher variance
- **High NW** (e.g., 4): 7 tapers, smoother, lower variance, broader peaks

> **Think about it:** For detecting a narrow oscillatory peak (like theta at
> ~8 Hz), would you prefer a low or high NW?  What about for estimating
> the broadband 1/f slope?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Compare multitaper PSDs with different NW
# ──────────────────────────────────────────────────────
# 1. Call psd_multitaper(x_long, fs=fs, nw=...) for NW = 2, 3, 4
# 2. Plot all three on the same axes (semilogy)
# 3. Also overlay the Welch PSD for comparison
# 4. Add a legend showing NW and number of tapers
# ──────────────────────────────────────────────────────

nw_values = [2, 3, 4]

raise NotImplementedError("Compute and compare multitaper PSDs with different NW")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

The number of tapers for a given NW is `K = int(2 * nw - 1)`.
Loop over the NW values, compute the PSD for each, and plot them
on the same figure.  A log-scale y-axis will compress the large
dynamic range of neural spectra.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
freqs_mt, psd_mt = psd_multitaper(x_long, fs=fs, nw=3.0)
```

`psd_multitaper` returns `(freqs, psd)` — note freqs come first.
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(freqs_welch, psd_welch, 'k--', alpha=0.5, label='Welch')

for nw in nw_values:
    freqs_mt, psd_mt = psd_multitaper(x_long, fs=fs, nw=nw)
    n_tapers = int(2 * nw - 1)
    ax.semilogy(freqs_mt, psd_mt, linewidth=0.8,
                label=f'NW={nw} ({n_tapers} tapers)')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (a.u.² / Hz)')
ax.set_title('Effect of NW on Multitaper PSD')
ax.set_xlim(0, 200)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 3
# Verify at least one multitaper PSD was computed
freqs_check, psd_check = psd_multitaper(x_long, fs=fs, nw=3.0)
assert len(freqs_check) > 10, "Too few frequency bins"
assert np.all(psd_check >= 0), "PSD must be non-negative"
print(f"Checkpoint 3 passed — Multitaper PSD: {len(freqs_check)} freq bins")

---
## Section 4 — Identifying Spectral Features

Now that you have a clean PSD estimate, annotate the key features.
Hippocampal LFP typically shows a 1/f background with oscillatory
bumps at characteristic frequencies.

> **Think about it:** The 1/f trend means low frequencies always have
> higher power.  How can you tell whether a peak at 8 Hz is a *real*
> oscillation vs just the 1/f background?

In [ ]:
# ──────────────────────────────────────────────────────
# YOUR TURN: Annotate the PSD with frequency bands
# ──────────────────────────────────────────────────────
# 1. Plot the multitaper PSD (NW=3) on a log scale
# 2. Use ax.axvspan() to shade frequency bands:
#    - Theta:  6–10 Hz
#    - Gamma: 30–90 Hz
#    - Ripple: 100–250 Hz (if visible)
# 3. Add a legend with band names
# ──────────────────────────────────────────────────────

raise NotImplementedError("Create an annotated PSD plot")

<details>
<summary><b>Hint 1 — Conceptual</b></summary>

`ax.axvspan(flo, fhi, alpha=0.15, color='red', label='Theta')` shades
a vertical band on the plot.  This visually highlights frequency regions
of interest.  You can overlay multiple bands with different colors.
</details>

<details>
<summary><b>Hint 2 — API</b></summary>

```python
ax.axvspan(6, 10, alpha=0.15, color='red', label='Theta')
ax.axvspan(30, 90, alpha=0.10, color='blue', label='Gamma')
```
</details>

<details>
<summary><b>Hint 3 — Partial code</b></summary>

```python
freqs_best, psd_best = psd_multitaper(x_long, fs=fs, nw=3.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(freqs_best, psd_best, 'k-', linewidth=0.8)
ax.axvspan(6, 10, alpha=0.15, color='red', label='Theta')
ax.axvspan(30, 90, alpha=0.10, color='blue', label='Gamma')
ax.axvspan(100, 250, alpha=0.10, color='green', label='Ripple')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (a.u.² / Hz)')
ax.set_title('Annotated PSD — Channel 0')
ax.set_xlim(1, 300)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
```
</details>

In [ ]:
# Checkpoint 4
# Verify the PSD shows expected 1/f trend (power at 1 Hz > power at 100 Hz)
freqs_final, psd_final = psd_multitaper(x_long, fs=fs, nw=3.0)
idx_low = np.argmin(np.abs(freqs_final - 2))
idx_high = np.argmin(np.abs(freqs_final - 100))
assert psd_final[idx_low] > psd_final[idx_high], (
    "Expected 1/f trend: power at 2 Hz should exceed power at 100 Hz"
)
print(f"Checkpoint 4 passed — 1/f confirmed: "
      f"PSD(2 Hz) = {psd_final[idx_low]:.2e} > PSD(100 Hz) = {psd_final[idx_high]:.2e}")

---
## Reflection

1. **Resolution trade-off:** You varied NW from 2 to 4.  In what situations
   would you choose NW=2 over NW=4?  Can you think of a neuroscience scenario
   where the difference matters?

2. **Multiple channels:** You analyzed one channel.  If you computed the PSD
   for all 16 channels, would you expect the spectra to be similar or different?
   Why? (Think about electrode placement on the shank.)

3. **Stationarity:** We computed the PSD over the entire recording.  Neural
   activity changes over time (sleep states, behavior).  How might non-stationarity
   affect your PSD estimate?  (This motivates the **spectrogram** in Exercise 4.)